In [ ]:
!pip install pandas librosa soundfile transformers peft tqdm scikit-learn
!pip install -q transformers accelerate peft librosa soundfile datasets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import io, os, librosa, soundfile as sf, numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoFeatureExtractor, AutoModel
from torch.optim import AdamW
from peft import PeftModel
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score
from transformers import AutoTokenizer, AutoFeatureExtractor, AutoModel

class CONFIG:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    LORA_PATH = "/content/drive/MyDrive/LORA Model"
    BASE_TEXT_MODEL = "j-hartmann/emotion-english-distilroberta-base"
    AUDIO_MODEL_NAME = "BilalHasan/distilhubert-finetuned-ravdess"
    PARQUET_PATH = "/content/drive/MyDrive/IEMOCAP 4 Class/Combined+Shuffled 4 Class 16KHz.parquet"
    SAVE_DIR = "/content/drive/MyDrive/IEMOCAP 4 Class/Full_MISA_MultiSampe/"

    BATCH_SIZE = 4
    WARMUP_EPOCHS = 3
    SOTA_EPOCHS = 12
    FOLDS = 5

    LR_WARMUP = 1e-4
    LR_BACKBONES = 1e-6
    LR_FUSION = 2e-5

    FEAT_DIM = 256
    NUM_CLASSES = 4
    LABEL_MAP = {'frustrated': 0, 'angry': 0, 'happy': 1, 'excited': 1, 'neutral': 2, 'sad': 3}

if not os.path.exists(CONFIG.SAVE_DIR): os.makedirs(CONFIG.SAVE_DIR)

class IEMOCAP4ClassDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.tokenizer = AutoTokenizer.from_pretrained(CONFIG.LORA_PATH)
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(CONFIG.AUDIO_MODEL_NAME)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with io.BytesIO(row['audio']['bytes']) as f:
            audio, sr = sf.read(f)
        if len(audio.shape) > 1: audio = np.mean(audio, axis=1)
        if sr != 16000: audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        a_in = self.feature_extractor(audio, sampling_rate=16000, return_tensors="pt", padding='max_length', max_length=160000, truncation=True).input_values.squeeze(0)
        t_in = self.tokenizer(str(row['transcription']), padding='max_length', max_length=128, truncation=True, return_tensors="pt")
        return {'input_ids': t_in['input_ids'].squeeze(0), 'attention_mask': t_in['attention_mask'].squeeze(0), 'audio_values': a_in, 'label': torch.tensor(CONFIG.LABEL_MAP[row['major_emotion'].lower()])}

class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, dim//2), nn.Tanh(), nn.Linear(dim//2, 1))
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return torch.sum(x * w, dim=1)

class MultiSampleDropoutHead(nn.Module):
    def __init__(self, feat_dim, num_classes, num_samples=5):
        super().__init__()
        self.num_samples = num_samples
        self.dropouts = nn.ModuleList([nn.Dropout(0.5) for _ in range(num_samples)])
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        logits = []
        for d in self.dropouts:
            logits.append(self.classifier(d(x)))
        return torch.stack(logits, dim=0).mean(dim=0)

class CrossModalAlignment(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, 8, batch_first=True)
        self.norm = nn.LayerNorm(dim)
    def forward(self, q, kv):
        attn_out, _ = self.attn(q, kv, kv)
        return self.norm(q + attn_out)

class ReliabilityGating(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(dim * 3, 128), nn.GELU(), nn.Linear(128, 3), nn.Softmax(dim=-1))
    def forward(self, pt, s, pa):
        return self.gate(torch.cat([pt, s, pa], dim=-1))

class MISA_GMC_Dropout(nn.Module):
    def __init__(self):
        super().__init__()
        t_base = AutoModel.from_pretrained(CONFIG.BASE_TEXT_MODEL)
        self.text_encoder = PeftModel.from_pretrained(t_base, CONFIG.LORA_PATH)
        self.audio_encoder = AutoModel.from_pretrained(CONFIG.AUDIO_MODEL_NAME)

        self.shared_proj = nn.Linear(768, CONFIG.FEAT_DIM)
        self.private_t_proj = nn.Linear(768, CONFIG.FEAT_DIM)
        self.private_a_proj = nn.Linear(768, CONFIG.FEAT_DIM)

        self.alignment = CrossModalAlignment(CONFIG.FEAT_DIM)
        self.gate = ReliabilityGating(CONFIG.FEAT_DIM)
        self.pool = AttentionPooling(CONFIG.FEAT_DIM)

        l = nn.TransformerEncoderLayer(d_model=CONFIG.FEAT_DIM, nhead=8, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(l, num_layers=2)
        self.classifier = MultiSampleDropoutHead(CONFIG.FEAT_DIM, CONFIG.NUM_CLASSES)

    def forward(self, input_ids, attention_mask, audio_values):
        t_h = self.text_encoder.model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        a_h = self.audio_encoder(audio_values).last_hidden_state

        st_h, sa_h = self.shared_proj(t_h), self.shared_proj(a_h)
        pt_p, pa_p = self.pool(self.private_t_proj(t_h)), self.pool(self.private_a_proj(a_h))

        s_aligned = self.alignment(st_h, sa_h)
        s_p = self.pool(s_aligned)

        w = self.gate(pt_p, s_p, pa_p)
        h = torch.stack([w[:,0:1]*pt_p, w[:,1:2]*s_p, w[:,2:3]*pa_p], dim=1)

        fused = self.fusion_transformer(h).mean(dim=1)
        return self.classifier(fused), pt_p, s_p, pa_p

def validate(model, loader):
    model.eval()
    ps, ts = [], []
    with torch.no_grad():
        for b in loader:
            logits, _, _, _ = model(b['input_ids'].to(CONFIG.device), b['attention_mask'].to(CONFIG.device), b['audio_values'].to(CONFIG.device))
            ps.extend(torch.argmax(logits, dim=1).cpu().numpy())
            ts.extend(b['label'].numpy())
    return accuracy_score(ts, ps)*100, recall_score(ts, ps, average='macro')

def load_and_transfer(model, path):
    if not os.path.exists(path): return model
    old_sd = torch.load(path, map_location=CONFIG.device)
    new_sd = model.state_dict()
    matched = {k: v for k, v in old_sd.items() if k in new_sd and "classifier" not in k and v.size() == new_sd[k].size()}
    new_sd.update(matched)
    model.load_state_dict(new_sd)
    print(f"Transferred {len(matched)} layers. Ready for Accuracy-focused training.")
    return model

def run_training():
    df = pd.read_parquet(CONFIG.PARQUET_PATH)
    df = df[df['major_emotion'].str.lower().isin(CONFIG.LABEL_MAP.keys())].copy()
    skf = StratifiedKFold(n_splits=CONFIG.FOLDS, shuffle=True, random_state=42)
    y_strat = df['major_emotion'].str.lower().map(CONFIG.LABEL_MAP).values
        train_loader = DataLoader(IEMOCAP4ClassDataset(df.iloc[t_idx]), batch_size=CONFIG.BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(IEMOCAP4ClassDataset(df.iloc[v_idx]), batch_size=CONFIG.BATCH_SIZE)

        model = MISA_GMC_Dropout().to(CONFIG.device)
        model = load_and_transfer(model, CONFIG.PREV_MODEL_PATH)

        print(f"[Phase 1] Warmup: Optimizing Fusion & Dropout...")
        for param in model.text_encoder.parameters(): param.requires_grad = False
        for param in model.audio_encoder.parameters(): param.requires_grad = False
        optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG.LR_WARMUP)

        for ep in range(CONFIG.WARMUP_EPOCHS):
            model.train()
            for b in tqdm(train_loader, desc=f"Warmup Ep {ep+1}", leave=False):
                optimizer.zero_grad()
                logits, _, _, _ = model(b['input_ids'].to(CONFIG.device), b['attention_mask'].to(CONFIG.device), b['audio_values'].to(CONFIG.device))
                loss = F.cross_entropy(logits, b['label'].to(CONFIG.device), label_smoothing=0.1)
                loss.backward()
                optimizer.step()
            acc, uar = validate(model, val_loader)
            print(f"Warmup Ep {ep+1} | Acc: {acc:.2f}% | UAR: {uar:.4f}")

        print(f"[Phase 2] Fine-tuning: Full Model Gradient Flow...")
        for param in model.parameters(): param.requires_grad = True
        optimizer = AdamW([
            {'params': model.text_encoder.parameters(), 'lr': CONFIG.LR_BACKBONES},
            {'params': model.audio_encoder.parameters(), 'lr': CONFIG.LR_BACKBONES},
            {'params': [p for n,p in model.named_parameters() if 'encoder' not in n], 'lr': CONFIG.LR_FUSION}
        ])

        best_acc = 0
        for ep in range(CONFIG.SOTA_EPOCHS):
            model.train()
            for b in tqdm(train_loader, desc=f"SOTA Ep {ep+1}", leave=False):
                optimizer.zero_grad()
                logits, pt, s, pa = model(b['input_ids'].to(CONFIG.device), b['attention_mask'].to(CONFIG.device), b['audio_values'].to(CONFIG.device))
                ce_loss = F.cross_entropy(logits, b['label'].to(CONFIG.device), label_smoothing=0.1)
                diff = (torch.norm(torch.matmul(pt, s.t())) + torch.norm(torch.matmul(pa, s.t()))) * 0.01
                (ce_loss + diff).backward()
                optimizer.step()

            acc, uar = validate(model, val_loader)
            print(f"SOTA Ep {ep+1} | Acc: {acc:.2f}% | UAR: {uar:.4f}")

            if acc > best_acc:
                best_acc = acc
                save_name = f"ms_dropout_fold_{fold+1}_best_acc.pt"
                torch.save(model.state_dict(), os.path.join(CONFIG.SAVE_DIR, save_name))
                print(f"⭐ New Best Acc: {best_acc:.2f}% - Model Saved")

if __name__ == "__main__":
    run_training()


>>> SKIPPING FOLD 1 (Already Trained)

>>> SKIPPING FOLD 2 (Already Trained)

>>> SKIPPING FOLD 3 (Already Trained)

==================== FOLD 4 / 5 ====================


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | UNEXPECTED | 
classifier.dense.weight         | UNEXPECTED | 
classifier.out_proj.bias        | UNEXPECTED | 
classifier.dense.bias           | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['id2label', 'label2id'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved

Loading weights:   0%|          | 0/49 [00:00<?, ?it/s]

HubertModel LOAD REPORT from: BilalHasan/distilhubert-finetuned-ravdess
Key               | Status     |  | 
------------------+------------+--+-
projector.weight  | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 
projector.bias    | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

SOTA Ep 1 | Acc: 80.55% | UAR: 0.8029
⭐ New Best Acc: 80.55% - Model Saved


SOTA Ep 2 | Acc: 82.34% | UAR: 0.8011
⭐ New Best Acc: 82.34% - Model Saved


SOTA Ep 3 | Acc: 82.39% | UAR: 0.7966
⭐ New Best Acc: 82.39% - Model Saved


SOTA Ep 4 | Acc: 82.08% | UAR: 0.8141


SOTA Ep 5 | Acc: 81.27% | UAR: 0.7883
